# Lecture 6: Sequence Processing & Tagging — Answer Key
### NLP Course 2027 — Week 03

---
Complete answers for all four exercises in Lecture 6.

In [ ]:
import nltk
from nltk import pos_tag, word_tokenize, FreqDist, ne_chunk
from nltk import DefaultTagger, UnigramTagger, BigramTagger, TrigramTagger
from nltk import RegexpParser
from nltk.corpus import treebank, brown
from collections import defaultdict

for pkg in ['averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng',
            'treebank', 'maxent_ne_chunker', 'maxent_ne_chunker_tab',
            'words', 'punkt', 'brown']:
    nltk.download(pkg, quiet=True)
print('Ready')

---
## Exercise 6.1 — POS Frequency Analysis

**Task:** `pos_frequency_analysis(text)` returns a FreqDist of POS tags. Apply to a news article.

**Key concept:** POS tag distributions reveal genre. News has many proper nouns (NNP) and past-tense verbs (VBD); fiction has more pronouns (PRP) and present-tense verbs (VBZ).

In [ ]:
def pos_frequency_analysis(text):
    """Return a FreqDist of POS tags in text."""
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)
    return FreqDist(tag for _, tag in tagged)

# Map Penn tags to readable names
TAG_NAMES = {
    'NN': 'Noun(sg)', 'NNS': 'Noun(pl)', 'NNP': 'ProperNoun(sg)', 'NNPS': 'ProperNoun(pl)',
    'VB': 'Verb(base)', 'VBD': 'Verb(past)', 'VBG': 'Verb(gerund)', 'VBN': 'Verb(pastpart)',
    'VBP': 'Verb(pres)', 'VBZ': 'Verb(3sg)', 'JJ': 'Adjective', 'RB': 'Adverb',
    'IN': 'Preposition', 'DT': 'Determiner', 'PRP': 'Pronoun', 'CC': 'Conjunction',
}

news_sample = """
The Federal Reserve raised interest rates by 0.25 percent on Wednesday.
The decision was unanimous among board members. Markets fell sharply.
Investors reacted with concern as Treasury yields climbed to new highs.
The central bank cited persistent inflation as the primary reason for the increase.
"""

pos_dist = pos_frequency_analysis(news_sample)
print('Top POS tags in news sample:')
print(f'{"Tag":<8} {"Name":<20} {"Count":>6}')
print('-' * 36)
for tag, count in pos_dist.most_common(12):
    name = TAG_NAMES.get(tag, tag)
    print(f'{tag:<8} {name:<20} {count:>6}')

---
## Exercise 6.2 — Trigram Tagger

**Task:** Build TrigramTagger (backoff chain: Trigram → Bigram → Unigram → Default). Compare all four taggers.

**Key concept:** N-gram taggers use context: a TrigramTagger looks at the previous two tags. Backoff handles unseen n-grams by falling back to lower-order taggers. Each level adds ~3–5% accuracy.

In [ ]:
tagged_sents = treebank.tagged_sents()
train_sents  = tagged_sents[:3000]
test_sents   = tagged_sents[3000:]

# Build backoff chain
t0 = DefaultTagger('NN')                          # baseline
t1 = UnigramTagger(train_sents,  backoff=t0)      # uses single word
t2 = BigramTagger(train_sents,   backoff=t1)      # uses 1 previous tag
t3 = TrigramTagger(train_sents,  backoff=t2)      # uses 2 previous tags

print(f'{"Tagger":<16} {"Accuracy":>10}')
print('-' * 28)
for name, tagger in [('DefaultTagger', t0), ('UnigramTagger', t1),
                     ('BigramTagger',  t2), ('TrigramTagger', t3)]:
    acc = tagger.accuracy(test_sents)
    print(f'{name:<16} {acc:>10.4f}')

print()
print('Each level of n-gram context adds ~3-5% accuracy.')
print('Trigram accuracy typically reaches ~89-91% on Treebank.')

---
## Exercise 6.3 — VP Chunker on Brown Corpus

**Task:** Write a VP grammar, extract VP chunks from Brown news, find most frequent patterns.

**Key concept:** VP chunking groups verb sequences. The pattern `<MD>?<VB.*>+<RB>?` captures modals + main verbs. Frequency analysis of VP patterns reveals the most common predicate structures in a corpus.

In [ ]:
grammar = r"""
  NP: {<DT>?<JJ.*>*<NN.*>+}
  VP: {<MD>?<VB.*>+<RB>?}
"""
chunker = RegexpParser(grammar)

# Extract VP chunks from Brown news sentences (first 200)
news_sents = brown.tagged_sents(categories='news')[:200]
vp_patterns = FreqDist()

for sent in news_sents:
    tree = chunker.parse(sent)
    for subtree in tree.subtrees(filter=lambda t: t.label() == 'VP'):
        # Store the POS tag pattern, not the words
        pattern = ' '.join(tag for _, tag in subtree.leaves())
        vp_patterns[pattern] += 1

print('Most frequent VP patterns in Brown news:')
print(f'{"Pattern":<25} {"Count":>6}')
print('-' * 33)
for pattern, count in vp_patterns.most_common(10):
    print(f'{pattern:<25} {count:>6}')

---
## Exercise 6.4 — NER Entity Extraction

**Task:** Extract all named entities from an Einstein article and organize by type.

**Key concept:** NLTK's `ne_chunk` uses a MaxEnt classifier trained on ACE data. It identifies PERSON, ORGANIZATION, GPE (geo-political entity), LOCATION, FACILITY. Organizing by type enables downstream tasks like knowledge graph construction.

In [ ]:
article = """
Albert Einstein was a German-born theoretical physicist who developed the theory of
relativity. Born in Ulm, Germany in 1879, Einstein moved to Switzerland and later
to the United States. He worked at Princeton University and received the Nobel Prize
in Physics in 1921. His famous equation E=mc2 changed the world of science.
"""

entities_by_type = defaultdict(set)

for sent in nltk.sent_tokenize(article):
    tokens = word_tokenize(sent)
    tagged = pos_tag(tokens)
    ne_tree = ne_chunk(tagged)
    for subtree in ne_tree:
        if hasattr(subtree, 'label'):
            entity = ' '.join(w for w, _ in subtree.leaves())
            entities_by_type[subtree.label()].add(entity)

print('Named Entities by Type:')
for etype, ents in sorted(entities_by_type.items()):
    print(f'  [{etype}]: {sorted(ents)}')

---
## Summary

| Exercise | Key Concept |
|----------|-------------|
| 6.1 | POS tag distribution reveals genre; news heavy on NNP and VBD |
| 6.2 | Backoff chain: each n-gram level adds ~3–5% accuracy on Treebank |
| 6.3 | VP chunking captures predicate structure; modal+verb patterns dominate news |
| 6.4 | NLTK NER identifies PERSON/GPE/ORG; organize by type for downstream use |

---
*NLP Course 2027 — Week 03*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**